## Cell 1 — Imports

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders
from src.training.trainer import ChagasTrainer

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('GPU required — switch runtime to GPU')

gpu_name = torch.cuda.get_device_name(0)
gpu_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  ({gpu_gb:.1f} GB)')
if gpu_gb < 5.5:
    raise RuntimeError(f'Need ≥6 GB GPU, found {gpu_gb:.1f} GB')


## Cell 2 — Configuration

In [ ]:
# ── QUICK TEST ──────────────────────────────────────────────────────────────
QUICK_TEST = False  # True = 5-min smoke test | False = full ~11h training

# ── FOLD ────────────────────────────────────────────────────────────────────
FOLD = 2   # 0–4

# ── PATHS ───────────────────────────────────────────────────────────────────
DATA_DIR       = project_root / 'data' / 'processed'
METADATA_CSV   = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR     = DATA_DIR / '2d_images'
SIGNALS_DIR    = DATA_DIR / '1d_signals_100hz'
CHECKPOINT_DIR = project_root / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MAE_CHECKPOINT   = CHECKPOINT_DIR / 'mae_2d_pretrained.pt'
STMEM_CHECKPOINT = CHECKPOINT_DIR / 'stmem_1d_pretrained.pt'

assert METADATA_CSV.exists(),     f'Missing: {METADATA_CSV}'
assert MAE_CHECKPOINT.exists(),   f'Missing: {MAE_CHECKPOINT} — run MAE pretraining first'
assert STMEM_CHECKPOINT.exists(), f'Missing: {STMEM_CHECKPOINT} — run ST-MEM pretraining first'

# ── HARDWARE ────────────────────────────────────────────────────────────────
# Phase 1: FM frozen → accum=4, eff.batch=64 (exact match to paper)
# Phase 2: all 173M unfrozen → accum=2, eff.batch=32 (GPU constraint)
BATCH_SIZE        = 16    # DO NOT change — Phase 2 OOM at 32 on 6 GB GPU
PHASE1_GRAD_ACCUM = 4     # eff.batch = 16 × 4 = 64  ✓ paper
PHASE2_GRAD_ACCUM = 2     # eff.batch = 16 × 2 = 32
NUM_WORKERS       = 2
USE_AMP           = True

# ── ITERATION SCALING ───────────────────────────────────────────────────────
# Van Santvliet: eff.batch=64 × 12,000 Phase-2 iters = 768,000 eff-samples
# v10.1 bug:     eff.batch=32 × 12,000              = 384,000  ← HALF
# v11 fix:       eff.batch=32 × 24,000              = 768,000  ✓ MATCHES
#
# Phase 1 unchanged: eff.batch=64 × 2,000 = 128,000  (same as paper)
PHASE1_ITERATIONS = 2000
PHASE2_ITERATIONS = 24000   # scaled: 12,000 × (64/32) = 24,000

PHASE1_LR      = 2e-4
PHASE2_LR_HIGH = 2e-4   # classifier + REPA head
PHASE2_LR_LOW  = 2e-5   # FM + 2D-ViT (discriminative fine-tuning)
MAX_GRAD_NORM  = 1.0
WARMUP_ITERS   = 200    # optimizer steps — NOT scaled with eff.batch

# ── VALIDATION ──────────────────────────────────────────────────────────────
# Scale VAL_EVERY to keep 3 mid-Phase-2 checks:
#   Phase 1: 2000 / 8000 = 0 checks  (blind; FM frozen; acceptable)
#   Phase 2: 24000 / 8000 = 3 checks  (at iters 8000, 16000, 24000)
VAL_EVERY = 1000

# ── AUGMENTATION — PAPER-ALIGNED ────────────────────────────────────────────
# Van Santvliet Table 1:  powerline prob=0.5, freq random from {50,60} Hz,
#                         SNR [15,30] dB, harmonics at SNR/2 and SNR/3
# augmentations.py DEFAULT_AUGMENTATION_CONFIG uses prob=0.3, freq=60 fixed.
# We override here to match the paper.
PAPER_AUGMENTATION_CONFIG = {
    'lead_mixup': {
        'prob': 0.3,        # Kim et al.: applied per pair independently
        'alpha': 0.2,
    },
    'powerline_noise': {
        'prob': 0.5,        # Van Santvliet: prob=0.5
        'use_snr': True,    # SNR-based amplitude (see augmentations_v2.py)
        'snr_range': (15, 30),   # dB — from Table 1
        'random_freq': True,     # randomly 50 or 60 Hz per sample
        'add_harmonics': True,   # 2nd + 3rd harmonic as per paper
    },
    'random_shift': {
        'prob': 0.5,        # Table 1: applied to all samples
        'max_shift': 100,   # ±100 samples @ 100 Hz = ±1s  (Table 1 ±1s)
    },
    'amplitude_scaling': {
        'prob': 0.3,
        'scale_range': (0.8, 1.2),
    },
    'baseline_wander': {
        'prob': 0.2,
        'amplitude': 0.2,
        'freq_range': (0.1, 0.5),
    },
}

# ── QUICK TEST OVERRIDES ────────────────────────────────────────────────────
if QUICK_TEST:
    PHASE1_ITERATIONS = 25
    PHASE2_ITERATIONS = 50
    WARMUP_ITERS      = 10
    VAL_EVERY         = 50
    NUM_WORKERS       = 0

# ── AUTO-RESUME ─────────────────────────────────────────────────────────────
_ckpt = CHECKPOINT_DIR / f'fold{FOLD}_latest.pt'
RESUME_FROM = str(_ckpt) if _ckpt.exists() else None

# ── SUMMARY ─────────────────────────────────────────────────────────────────
p1 = BATCH_SIZE * PHASE1_GRAD_ACCUM * PHASE1_ITERATIONS
p2 = BATCH_SIZE * PHASE2_GRAD_ACCUM * PHASE2_ITERATIONS
paper_p2 = 64 * 12000
p2_match = '✓' if p2 == paper_p2 else f'✗ (paper={paper_p2:,})'

print(f'Fold {FOLD}  |  QUICK_TEST={QUICK_TEST}')
print(f'Batch {BATCH_SIZE} | P1 eff.batch={BATCH_SIZE*PHASE1_GRAD_ACCUM} | P2 eff.batch={BATCH_SIZE*PHASE2_GRAD_ACCUM}')
print(f'Phase 1: {PHASE1_ITERATIONS} iters  → {p1:,} eff-samples')
print(f'Phase 2: {PHASE2_ITERATIONS} iters  → {p2:,} eff-samples  {p2_match}')
print(f'VAL_EVERY={VAL_EVERY}  |  Resume: {RESUME_FROM or "fresh start"}')
if not QUICK_TEST:
    print(f'ETA: Phase 1 ~18 min  |  Phase 2 ~10–12 h  |  Total ~11 h')


## Cell 3 — GPU Memory Check

In [ ]:
torch.cuda.empty_cache()
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
free_gb  = total_gb - torch.cuda.memory_reserved() / 1e9
print(f'VRAM: {total_gb:.1f} GB total | {free_gb:.2f} GB free')
if free_gb < 3.5:
    raise RuntimeError(f'Only {free_gb:.1f} GB free — restart kernel or free GPU memory')
print('Memory check passed.')


## Cell 4 — Dataloaders  (paper-aligned augmentation config)

In [ ]:
# create_dataloaders() was updated to accept augmentation_config.
# If your dataset.py doesn't have this parameter yet, use the updated version.
# The PAPER_AUGMENTATION_CONFIG passes powerline prob=0.5, random 50/60Hz,
# harmonics — matching Van Santvliet Table 1 exactly.

train_loader, val_loader = create_dataloaders(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    fold=FOLD,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampling=True,   # 5× oversample positives (Van Santvliet)
    augment_train=True,
    augmentation_config=PAPER_AUGMENTATION_CONFIG,  # v11: paper-aligned
)

# Sanity-check
batch = next(iter(train_loader))
assert batch['image'].shape  == torch.Size([BATCH_SIZE, 3, 24, 2048]), f"Image shape: {batch['image'].shape}"
assert batch['signal'].shape == torch.Size([BATCH_SIZE, 12, 1000]),   f"Signal shape: {batch['signal'].shape}"
assert not torch.isnan(batch['signal']).any(), 'NaN in signals'
assert not torch.isnan(batch['image'].float()).any(), 'NaN in images'

unique_labels = sorted({round(v, 1) for v in batch['label'].tolist()})
assert set(unique_labels) <= {0.0, 0.2, 0.8, 1.0}, f'Bad labels: {unique_labels}'

n_train   = len(train_loader.dataset)
n_pos     = int(train_loader.dataset.df['label_hard'].sum())
n_val     = len(val_loader.dataset)
n_pos_val = int(val_loader.dataset.df['label_hard'].sum())
print(f'Train: {n_train:,} ({n_pos:,} pos = {100*n_pos/n_train:.2f}%)')
print(f'Val:   {n_val:,}  ({n_pos_val:,} pos = {100*n_pos_val/n_val:.2f}%)')
print(f'Label values in batch: {unique_labels}')


## Cell 5 — Model + Pretrained Weights

In [ ]:
model = HybridChagasModel(
    img_size=(24, 2048), patch_size_2d=(8, 64),
    num_leads=12, seq_len_1d=1000, patch_size_1d=50,
    embed_dim=768, depth=12, num_heads=12,
    use_aol=True, use_demographics=True,
)

# flatten_dict fix ensures 145+ transformer keys load (not just 5-6 top-level)
model.vit_2d.load_mae_pretrained(str(MAE_CHECKPOINT))
model.vit_1d_fm.load_stmem_pretrained(str(STMEM_CHECKPOINT))

model = model.to(device)

with torch.no_grad():
    out = model(batch['image'].to(device), batch['signal'].to(device),
                batch['age'].to(device),   batch['sex'].to(device))
assert torch.isfinite(out['logits']).all(),      'Non-finite logits'
assert torch.isfinite(out['fm_features']).all(), 'Non-finite FM features'

total_p = sum(p.numel() for p in model.parameters())
print(f'Params: {total_p:,}  |  logits: {out["logits"].shape}  |  FM: {out["fm_features"].shape}')


## Cell 6 — Trainer

In [ ]:
# CRITICAL: pass phase2_grad_accum=2 explicitly.
# trainer.py default is phase2_grad_accum=1 → eff.batch=16 if not overridden.
# trainer.py ETA for Phase 2 = self.phase2_iterations * 2.5 / 3600
#   With 24,000 iters: 24000 * 2.5 / 3600 = 16.7h (overestimates; real ~10-12h)
#   This is acceptable — it is an upper-bound estimate.

trainer = ChagasTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    phase1_iterations=PHASE1_ITERATIONS,   # 2,000
    phase2_iterations=PHASE2_ITERATIONS,   # 24,000  ← scaled
    phase1_lr=PHASE1_LR,                   # 2e-4
    phase2_lr_high=PHASE2_LR_HIGH,         # 2e-4
    phase2_lr_low=PHASE2_LR_LOW,           # 2e-5
    checkpoint_dir=str(CHECKPOINT_DIR),
    use_amp=USE_AMP,
    max_grad_norm=MAX_GRAD_NORM,           # 1.0
    warmup_iters=WARMUP_ITERS,             # 200 (NOT scaled)
    phase1_grad_accum=PHASE1_GRAD_ACCUM,   # 4 → eff.batch 64
    phase2_grad_accum=PHASE2_GRAD_ACCUM,   # 2 → eff.batch 32  ← EXPLICIT
    val_every_n_iters=VAL_EVERY,           # 8,000  ← scaled
    val_subset_size=300  if QUICK_TEST else 3000,
    val_n_permutations=100 if QUICK_TEST else 1000,
)

p2_checks = PHASE2_ITERATIONS // VAL_EVERY
print(f'Val every {VAL_EVERY} iters | P1 checks: 0 | P2 checks: {p2_checks}')
print(f'Val subset {trainer.val_subset_size} | Perms {trainer.val_n_permutations}')
if not QUICK_TEST:
    print('ETA: Phase 1 ~18 min | Phase 2 ~10–12 h')


## Cell 7 — Train

In [ ]:
if RESUME_FROM:
    print(f'Resuming from: {Path(RESUME_FROM).name}')
else:
    print('Starting fresh')

metrics = trainer.train(fold=FOLD, resume_from=RESUME_FROM)

tpr = metrics['tpr_5pct']
print(f'\nFold {FOLD} results:')
print(f'  TPR@5%: {tpr:.4f}  (PRIMARY)')
print(f'  AUROC:  {metrics["auroc"]:.4f}')
print(f'  AUPRC:  {metrics.get("auprc", 0):.4f}')
print(f'  Method: {"OFFICIAL" if metrics.get("using_official") else "APPROXIMATE"}')

if not QUICK_TEST:
    for name, val in [
        ('Random baseline',                0.050),
        ('Kim 2025 hidden val',             0.369),
        ('Challenge target',               0.420),
        ('Van Santvliet val (top team)',    0.445),
        ('Van Santvliet CV mean',           0.490),
    ]:
        diff = tpr - val
        print(f'  {"↑" if diff >= 0 else "↓"}{abs(diff):.4f}  vs  {name} ({val:.3f})')


## Cell 8 — Save Results and Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

results_df = pd.DataFrame([metrics])
results_df['fold'] = FOLD
results_df['phase2_iters'] = PHASE2_ITERATIONS
results_df['phase2_eff_batch'] = BATCH_SIZE * PHASE2_GRAD_ACCUM
results_df.to_csv(CHECKPOINT_DIR / f'fold{FOLD}_results.csv', index=False)

history = trainer.history
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3)

ax0 = fig.add_subplot(gs[0])
if history['train_loss']:
    ax0.plot(history['train_loss'], lw=0.8, alpha=0.8, color='steelblue')
    if PHASE1_ITERATIONS < len(history['train_loss']):
        ax0.axvline(PHASE1_ITERATIONS, color='r', ls='--', lw=1.2,
                    label=f'Phase 2 start ({PHASE1_ITERATIONS})')
        ax0.legend(fontsize=8)
    ax0.set(xlabel='Iteration', ylabel='Loss', title='Training Loss')
    ax0.grid(True, alpha=0.3)

ax1 = fig.add_subplot(gs[1])
if history['val_tpr_5pct']:
    iters = [VAL_EVERY * (i + 1) for i in range(len(history['val_tpr_5pct']))]
    ax1.plot(iters, history['val_tpr_5pct'], 'go-', ms=5, lw=1.5)
    ax1.set(xlabel='Iteration', ylabel='TPR@5%', title='Val TPR@5%')
    ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[2])
if history['grad_norm']:
    g = history['grad_norm'][:PHASE1_ITERATIONS]
    ax2.plot(g, lw=0.6, alpha=0.6, color='darkorange')
    ax2.axhline(1.0, color='r', ls='--', lw=1.2, label='Clip @ 1.0')
    clipped = 100 * sum(1 for v in g if v > 1.0) / max(1, len(g))
    ax2.set(xlabel='Iteration', ylabel='Grad norm',
            title=f'Grad Norm Phase 1 ({clipped:.0f}% clipped)')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

fig.suptitle(
    f'Fold {FOLD}  |  P2 iters={PHASE2_ITERATIONS}  |  eff.batch={BATCH_SIZE*PHASE2_GRAD_ACCUM}',
    fontsize=12, fontweight='bold')
plt.tight_layout()
path = CHECKPOINT_DIR / f'fold{FOLD}_training_curve.png'
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show(); print(f'Saved: {path.name}')

## Cell 9 — Checkpoint Verification

In [ ]:
for p in sorted(CHECKPOINT_DIR.glob(f'fold{FOLD}*.pt')):
    print(f'  {p.name}  ({p.stat().st_size/1e6:.0f} MB)')

best = CHECKPOINT_DIR / f'fold{FOLD}_best.pt'
if best.exists():
    ckpt = torch.load(best, map_location='cpu', weights_only=False)
    vs = ckpt.get('val_score', None)
    score_str = f'{vs:.4f}' if vs is not None else 'n/a'   # ← fix here
    print(f'best: score={score_str}  phase={ckpt.get("phase","?")}  iter={ckpt.get("iteration","?")}')
    del ckpt

print('\nAll-fold status:')
for f in range(5):
    done = (CHECKPOINT_DIR / f'fold{f}_best.pt').exists()
    score = ''
    if done:
        try:
            c = torch.load(CHECKPOINT_DIR / f'fold{f}_best.pt', map_location='cpu', weights_only=False)
            s = c.get('val_score', None)
            score = f'  TPR@5%={s:.4f}' if s is not None else ''
            del c
        except: pass
    print(f'  [{"✓" if done else "○"}] Fold {f}{score}')

nxt = next((f for f in range(5) if not (CHECKPOINT_DIR / f'fold{f}_best.pt').exists()), None)
if nxt is None:
    print('\nAll 5 folds complete → run evaluation_complete_v3.1.ipynb')
else:
    print(f'\nNext: set FOLD={nxt} in Cell 2 and re-run')